# Table 1: AoU-LR cohort summary

Rebuild manuscript Table 1 from the person-level covariates table.

**Primary input** (first existing path wins)

- `$AOU_COVARIATES` if set
- `covariates.v6.csv.gz` at the workspace root
- `covariates.source_rebuilt.csv.gz` under the data root

PacBio median read length is `read_length_median` (companion to `coverage`:
PacBio-prefer technical collapse, Phase 1-only filled from Integratedcall).

ONT coverage and median read length are `ont_coverage` /
`ont_read_length_median` (`ont_metrics_source`: Phase 1 `ont_sample_hg38`,
Phase 2 `ont_v9_technical`). These never substitute for primary `coverage`.

**Stratum definitions**

| Column | Definition |
|---|---|
| Phase 1 mid-pass | `in_cdr_v7` (n = 1,027) |
| Phase 2 PacBio discovery | `final_releasable_v9` and `technology == PacBio` |
| Phase 2 high-pass | all discovery PacBio+ONT dual-tech samples, plus highest-coverage PacBio-only samples to n = 1,133 |
| Phase 2 mid-pass | remaining PacBio discovery samples |

An explicit mid/high-pass label is not stored in the covariates table. The
high-pass proxy keeps ONT overlap in the high-pass column.

**Pedigrees** are complete recorded families (`n == pedigree_family_size`)
whose members all fall in the Phase 2 PacBio discovery set. A family is
multi-generational when `pedigree_n_generations >= 3` (generation depth from
recorded parent IDs), not merely because it has five or more people. They are a
Phase 2 resource, not a mid/high-pass attribute, so they are omitted from
Table 1 and written as a supplementary table.

**Outputs**

- `summaries/manuscript/table1_cohort_summary.tsv`
- `summaries/manuscript/table1_cohort_summary.md`
- `summaries/manuscript/tableS_phase2_pedigrees.tsv`
- `summaries/manuscript/tableS_phase2_pedigrees.md`


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# On Terra, localize $WORKSPACE_BUCKET/scripts/ before importing anything.
# Persistent edit/scripts/ copies are often stale and must not win.
_bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
_sync = os.environ.get("TERRA_SYNC_SCRIPTS", "true" if _bucket else "").strip().lower() in {
    "1", "true", "yes", "on",
}
_scripts = None
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file() or (_d / "workspace_paths.py").is_file():
        _scripts = _d.resolve()
        break
if _bucket and _sync:
    if _scripts is None:
        _scripts = (Path.cwd() / "scripts").resolve()
    _scripts.mkdir(parents=True, exist_ok=True)
    print(f"gsutil -m rsync -r {_bucket}/scripts/ {_scripts}/")
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_scripts) + "/"])
    os.environ["TERRA_SCRIPTS_LOCALIZED"] = "true"
    import importlib
    importlib.invalidate_caches()
    _prefix = str(_scripts)
    for _name, _mod in list(sys.modules.items()):
        _file = getattr(_mod, "__file__", None)
        if _file and str(_file).startswith(_prefix):
            sys.modules.pop(_name, None)
elif _scripts is None:
    raise FileNotFoundError(
        "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
        "Upload scripts/ to gs://WORKSPACE/scripts/."
    )
sys.path.insert(0, str(_scripts))

from terra_notebook import init_notebook

SCRIPTS = init_notebook("workspace_paths.py")
from workspace_paths import WORKSPACE, data_root

import numpy as np
import pandas as pd

ROOT = data_root()


def resolve_covariates() -> Path:
    env = os.environ.get("AOU_COVARIATES")
    if env:
        path = Path(env).expanduser().resolve()
        if path.is_file():
            return path
        raise FileNotFoundError(f"AOU_COVARIATES is not a file: {path}")
    for path in (
        WORKSPACE / "covariates.v6.csv.gz",
        ROOT / "covariates.v6.csv.gz",
        ROOT / "covariates.source_rebuilt.csv.gz",
    ):
        if path.is_file():
            return path
    raise FileNotFoundError(
        "No covariates table found. Set AOU_COVARIATES or place "
        "covariates.v6.csv.gz / covariates.source_rebuilt.csv.gz on the data root."
    )


COV_CSV = resolve_covariates()

OUT_DIR = ROOT / "summaries" / "manuscript"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_TSV = OUT_DIR / "table1_cohort_summary.tsv"
OUT_MD = OUT_DIR / "table1_cohort_summary.md"

HIGH_PASS_N = 1133

assert COV_CSV.exists(), COV_CSV

print("ROOT:", ROOT)
print("COV_CSV:", COV_CSV)
print("OUT_DIR:", OUT_DIR)


In [ ]:
def mean_sd(series: pd.Series, digits: int = 1) -> str:
    x = pd.to_numeric(series, errors="coerce").dropna()
    if len(x) == 0:
        return "—"
    if len(x) == 1:
        return f"{x.iloc[0]:.{digits}f}"
    return f"{x.mean():.{digits}f} ± {x.std(ddof=1):.{digits}f}"


def mean_sd_int(series: pd.Series) -> str:
    x = pd.to_numeric(series, errors="coerce").dropna()
    if len(x) == 0:
        return "—"
    if len(x) == 1:
        return f"{x.iloc[0]:,.0f}"
    return f"{x.mean():,.0f} ± {x.std(ddof=1):,.0f}"


def overlap_label(n_overlap: int, n_total: int) -> str:
    if n_total == 0:
        return "—"
    if n_overlap == 0:
        return "0"
    return f"{n_overlap}/{n_total:,}"


In [ ]:
cov = pd.read_csv(
    COV_CSV,
    dtype={"research_id": str, "biobank_id": str, "pedigree_family_id": str},
    low_memory=False,
)
print(f"covariates: {len(cov):,} people from {COV_CSV.name}")
print(cov[["in_cdr_v7", "final_releasable_v9", "has_ONT", "has_PacBio"]].sum().to_string())
print("read_length_median_source:")
print(cov["read_length_median_source"].value_counts(dropna=False).to_string())


## Define Table 1 strata

Phase 2 high-pass is not an explicit covariates field. The proxy below matches the
manuscript pattern: ONT overlap sits in high-pass, mid-pass PacBio discovery has
no ONT overlap, and high-pass PacBio discovery size is 1,133.


In [ ]:
phase1 = cov.loc[cov["in_cdr_v7"]].copy()
pacbio = cov.loc[cov["final_releasable_v9"] & cov["technology"].eq("PacBio")].copy()
assert pacbio["coverage"].notna().all(), "missing coverage for releasable PacBio discovery samples"
assert pacbio["read_length_median"].notna().all(), "missing read_length_median for releasable PacBio discovery samples"

dual = pacbio.loc[pacbio["has_ONT"]].copy()
pacbio_only = pacbio.loc[~pacbio["has_ONT"]].copy()
n_fill = HIGH_PASS_N - len(dual)
assert n_fill >= 0, (HIGH_PASS_N, len(dual))
high = pd.concat(
    [dual, pacbio_only.nlargest(n_fill, "coverage")],
    ignore_index=True,
)
assert len(high) == HIGH_PASS_N
assert high["research_id"].is_unique

mid = pacbio.loc[~pacbio["research_id"].isin(set(high["research_id"]))].copy()
assert len(mid) + len(high) == len(pacbio)
assert int(mid["has_ONT"].sum()) == 0

strata = {
    "phase1_midpass": phase1,
    "phase2_midpass": mid,
    "phase2_highpass": high,
}

print({k: len(v) for k, v in strata.items()})
print("Phase 2 PacBio discovery:", len(pacbio))
print("high-pass ONT dual-tech:", int(high["has_ONT"].sum()))
print(
    "ONT-primary releasable samples not shown in PacBio-centric columns:",
    int((cov["final_releasable_v9"] & cov["technology"].eq("ONT")).sum()),
)


In [ ]:
def classify_families(covariates: pd.DataFrame) -> pd.DataFrame:
    ped = covariates.loc[covariates["in_pedigree"]].copy()
    rows = []
    for family_id, group in ped.groupby("pedigree_family_id"):
        ids = set(group["research_id"])
        n = len(group)
        n_founders = int(group["pedigree_is_founder"].fillna(False).astype(bool).sum())
        n_two_parents = int((pd.to_numeric(group["pedigree_n_parents"], errors="coerce") == 2).sum())
        size_col = pd.to_numeric(group["pedigree_family_size"], errors="coerce").iloc[0]
        n_gen = pd.to_numeric(group["pedigree_n_generations"], errors="coerce").iloc[0]
        if pd.isna(size_col) or n != int(size_col):
            kind = "incomplete"
        elif pd.notna(n_gen) and int(n_gen) >= 3:
            kind = "multi-generational"
        elif n == 3 and n_founders == 2 and n_two_parents == 1:
            kind = "trio"
        elif n == 4 and n_founders == 2 and n_two_parents == 2:
            kind = "quartet"
        else:
            kind = "other"
        rows.append({"family_id": family_id, "kind": kind, "ids": ids})
    return pd.DataFrame(rows)


FAMILIES = classify_families(cov)
print(FAMILIES["kind"].value_counts().to_string())


def stratum_column(stratum: pd.DataFrame, *, phase1: bool) -> dict[str, object]:
    if phase1:
        pb_coverage = stratum.loc[
            stratum["coverage_source"].eq("integratedcall_phase1"), "coverage"
        ]
        pb_rl = stratum.loc[
            stratum["read_length_median_source"].eq("integratedcall_phase1"),
            "read_length_median",
        ]
        ont = stratum.loc[stratum["ont_coverage"].notna()]
        ont_n = len(ont)
        ont_cov = ont["ont_coverage"]
        ont_rl = ont["ont_read_length_median"]
        # Resequenced Phase 1 participants contribute omics in Phase 2 only.
        meth = rna = prot = 0
    else:
        pb_coverage = stratum["coverage"]
        pb_rl = stratum["read_length_median"]
        ont_n = int(stratum["has_ONT"].sum())
        ont_cov = stratum.loc[stratum["has_ONT"], "ont_coverage"]
        ont_rl = stratum.loc[stratum["has_ONT"], "ont_read_length_median"]
        meth = int(stratum["has_methylation"].sum())
        rna = int(stratum["has_rna"].sum())
        prot = int(stratum["has_proteomics"].sum())

    return {
        "PacBio discovery participants": f"{len(stratum):,}",
        "PacBio mean coverage": mean_sd(pb_coverage),
        "PacBio median read length (bp)": mean_sd_int(pb_rl),
        "ONT overlapping participants": overlap_label(ont_n, len(stratum)),
        "ONT mean coverage": mean_sd(ont_cov) if ont_n and ont_cov.notna().any() else "—",
        "ONT median read length (bp)": mean_sd_int(ont_rl) if ont_n and ont_rl.notna().any() else "—",
        "Participants with CpG methylation": f"{meth:,}",
        "Participants with RNA-seq": f"{rna:,}",
        "Participants with Olink proteomics": f"{prot:,}",
        "Participants with EHRs": f"{int((stratum['has_ehr_data'] == True).sum()):,}",
    }


p2_ids = set(pacbio["research_id"])
mid_ids = set(mid["research_id"])
high_ids = set(high["research_id"])
COMPLETE_KINDS = ("trio", "quartet", "multi-generational")


def family_placement(ids: set[str]) -> str:
    if not ids <= p2_ids:
        return "not_all_discovery"
    n_mid = len(ids & mid_ids)
    n_high = len(ids & high_ids)
    if n_high == 0:
        return "mid_only"
    if n_mid == 0:
        return "high_only"
    return "mixed"


p2_families = FAMILIES.loc[
    FAMILIES["kind"].isin(COMPLETE_KINDS) & FAMILIES["ids"].map(lambda ids: ids <= p2_ids)
].copy()
p2_families["placement"] = p2_families["ids"].map(family_placement)
p2_families["n_people"] = p2_families["ids"].map(len)
p2_families["n_mid"] = p2_families["ids"].map(lambda ids: len(ids & mid_ids))
p2_families["n_high"] = p2_families["ids"].map(lambda ids: len(ids & high_ids))

ped_rows = []
for kind in COMPLETE_KINDS:
    sub = p2_families.loc[p2_families["kind"].eq(kind)]
    ped_rows.append({
        "Family type": kind,
        "Families": len(sub),
        "People": int(sub["n_people"].sum()) if len(sub) else 0,
        "Wholly mid-pass": int(sub["placement"].eq("mid_only").sum()),
        "Wholly high-pass": int(sub["placement"].eq("high_only").sum()),
        "Mixed mid/high": int(sub["placement"].eq("mixed").sum()),
    })
ped_table = pd.DataFrame(ped_rows)
ped_table.loc[len(ped_table)] = {
    "Family type": "Total",
    "Families": int(ped_table["Families"].sum()),
    "People": int(ped_table["People"].sum()),
    "Wholly mid-pass": int(ped_table["Wholly mid-pass"].sum()),
    "Wholly high-pass": int(ped_table["Wholly high-pass"].sum()),
    "Mixed mid/high": int(ped_table["Mixed mid/high"].sum()),
}
print("Phase 2 PacBio discovery pedigrees (complete families):")
try:
    display(ped_table)
except NameError:
    print(ped_table.to_string(index=False))
print(
    "people in complete pedigrees:",
    int(p2_families["n_people"].sum()),
    "mid-pass",
    int(p2_families["n_mid"].sum()),
    "high-pass",
    int(p2_families["n_high"].sum()),
)


In [ ]:
table = pd.DataFrame({
    "AoU-LR Phase 1: mid-pass (~8x)": stratum_column(strata["phase1_midpass"], phase1=True),
    "AoU-LR Phase 2: mid-pass (~15x)": stratum_column(strata["phase2_midpass"], phase1=False),
    "AoU-LR Phase 2: high-pass (~30x)": stratum_column(strata["phase2_highpass"], phase1=False),
})
table.index.name = "Metric"
table = table.reset_index()

section_for = {
    "PacBio discovery participants": "Sequencing platform — PacBio",
    "PacBio mean coverage": "Sequencing platform — PacBio",
    "PacBio median read length (bp)": "Sequencing platform — PacBio",
    "ONT overlapping participants": "Sequencing platform — ONT",
    "ONT mean coverage": "Sequencing platform — ONT",
    "ONT median read length (bp)": "Sequencing platform — ONT",
    "Participants with CpG methylation": "Multi-omic and EHR linkage",
    "Participants with RNA-seq": "Multi-omic and EHR linkage",
    "Participants with Olink proteomics": "Multi-omic and EHR linkage",
    "Participants with EHRs": "Multi-omic and EHR linkage",
}
table.insert(0, "Section", table["Metric"].map(section_for))

try:
    display(table)
except NameError:
    print(table.to_string(index=False))

table.to_csv(OUT_TSV, sep="\t", index=False)
ped_table.to_csv(OUT_PED_TSV, sep="\t", index=False)

p1_n = len(strata["phase1_midpass"])
mid_n = len(strata["phase2_midpass"])
high_n = len(strata["phase2_highpass"])
p2_n = mid_n + high_n

md_lines = [
    "# Table 1. AoU-LR cohort summary",
    "",
    f"Generated from `{COV_CSV.name}`.",
    "",
    "| Section | Metric | Phase 1 mid-pass (~8x) | Phase 2 mid-pass (~15x) | Phase 2 high-pass (~30x) |",
    "|---|---|---:|---:|---:|",
]
for _, row in table.iterrows():
    md_lines.append(
        f"| {row['Section']} | {row['Metric']} | "
        f"{row['AoU-LR Phase 1: mid-pass (~8x)']} | "
        f"{row['AoU-LR Phase 2: mid-pass (~15x)']} | "
        f"{row['AoU-LR Phase 2: high-pass (~30x)']} |"
    )
md_lines.extend([
    "",
    "## Definitions",
    "",
    f"- Phase 1: `in_cdr_v7` (n = {p1_n:,}).",
    f"- Phase 2 PacBio discovery: `final_releasable_v9` and `technology == PacBio` (n = {p2_n:,}).",
    f"- Phase 2 high-pass: all PacBio+ONT dual-tech discovery samples, plus highest-coverage PacBio-only samples to n = {HIGH_PASS_N:,}.",
    "- Phase 2 mid-pass: remaining PacBio discovery samples.",
    "- Phase 1 PacBio mean coverage and median read length use `*_source == integratedcall_phase1` (Phase 1-only samples).",
    "- Pedigrees are omitted from Table 1 (families are not a mid/high-pass attribute) and written as table S1.",
    "- Multi-omic flags for 14 Phase 1 participants resequenced in Phase 2 are counted in Phase 2 only.",
    "- Phase 2 ONT coverage / median read length use `ont_*` filled from the CDRv9 ONT_v9 sheet (`ont_metrics_source == ont_v9_technical`), plus one overlapping Phase 1 ONT sample that keeps the Phase 1 ONT sheet.",
    "- ONT-primary releasable samples are not shown in this PacBio-centric layout.",
    "",
])
OUT_MD.write_text("\n".join(md_lines))

ped_md = [
    "# Table S1. Complete pedigrees in the Phase 2 PacBio discovery set",
    "",
    f"Generated from `{COV_CSV.name}`.",
    "",
    "| Family type | Families | People | Wholly mid-pass | Wholly high-pass | Mixed mid/high |",
    "|---|---:|---:|---:|---:|---:|",
]
for _, row in ped_table.iterrows():
    ped_md.append(
        f"| {row['Family type']} | {row['Families']} | {row['People']} | "
        f"{row['Wholly mid-pass']} | {row['Wholly high-pass']} | {row['Mixed mid/high']} |"
    )
ped_md.extend([
    "",
    "Complete recorded families (`n == pedigree_family_size`) whose members all fall in the Phase 2 PacBio discovery set.",
    "Multi-generational means `pedigree_n_generations >= 3` from recorded parent IDs (both such families are UW high-pass dual-tech).",
    f"Of {int(p2_families['n_people'].sum())} people in these families, "
    f"{int(p2_families['n_mid'].sum())} are mid-pass and {int(p2_families['n_high'].sum())} are high-pass.",
    "Mixed families typically have one or more ONT dual-tech members assigned to the high-pass proxy.",
    "",
])
OUT_PED_MD.write_text("\n".join(ped_md))
print("wrote", OUT_TSV)
print("wrote", OUT_MD)
print("wrote", OUT_PED_TSV)
print("wrote", OUT_PED_MD)
